In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'models' / 'src'))

from common import contract, data, paths
from common.data import show_dataset


from common import plots
from common.plots import (SUPERFICIE, TINTA, TINTA_2, MUDO, GRADE, EIXO,
                          REAL, PREDITO, ERRO_CLARO, MAE_ESC, HALO, enfeitar)

plots.aplicar_estilo()


SEED = contract.SEED
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
NL = chr(10)   # evita barra invertida no fonte; usado nos relatorios


print(f"dados em {paths.PROCESSED}")
print(f"saida  em {paths.SPLITS}")
print(f"semente: {SEED}")

def get_split(dataset: pd.DataFrame, rate=0.8):
  train = dataset.sample(frac=rate, random_state=SEED)
  validation = dataset.drop(train.index)
  return train.index, validation.index


In [8]:
raw: pd.DataFrame = data.load_pool(paths.PROCESSED)

print(f"{raw.inversor.nunique()} arquivos" + NL)
print(raw.groupby("inversor").size().to_string())


3 arquivos

inversor
ifpee1    1763
ifpee2     696
ifpee3     695


In [9]:

df: pd.DataFrame = show_dataset(raw.copy())


pool: 3154 linhas, 12 colunas

nulos por coluna (so as que tem):
  (nenhum)

colunas com zero:
  (nenhum)


In [10]:
def year_seasons(md):
    if 320 <= md < 621:
        return 'outono'
    elif 621 <= md < 922:
        return 'inverno'
    elif 922 <= md < 1221:
        return 'primavera'
    else:
        return 'verão'

df['season'] = (
    df['datetime'].dt.month * 100 
    + df['datetime'].dt.day
).apply(year_seasons)

df.head()

,datetime,PCA,ghit,ioeff,jacques,kc,kt,pvsyst_cell,sandia,sandia_cell,temperatura_ambiente,inversor,season
0,2021-01-04,80.25,509.805506,1411.074617,42.760211,0.705158,0.472495,44.073776,42.401079,46.569444,29.834381,ifpee1,verão
1,2021-01-05,140.51,613.979487,1411.014645,43.841672,0.874327,0.599576,45.791870,43.430381,48.464121,28.642788,ifpee1,verão
2,2021-01-06,148.08,649.480247,1410.941382,47.091966,0.924150,0.637047,48.157670,46.614975,51.905905,30.017015,ifpee1,verão
3,2021-01-07,143.97,630.924510,1410.854850,46.378474,0.922133,0.631160,47.826783,45.936135,51.090729,30.204409,ifpee1,verão
4,2021-01-08,135.86,591.560793,1410.755074,45.315992,0.848405,0.588750,46.690435,44.914696,49.750472,30.167530,ifpee1,verão


In [18]:

# pegar somente p ultimo ano para teste
test_split = (
  (df.datetime.dt.year == df.datetime.dt.year.max()) & 
  (df.inversor == 'ifpee1'))

splits: dict[str, pd.Index] = {
  'test': df[test_split], 
  'train': df[~test_split], 
  'validation': df[~test_split],
}

# garantir pela menos uma estação do ano no split
primavera = (df['season'] == 'primavera')
inverno = (df['season'] == 'inverno')
outono = (df['season'] == 'outono')
verão = (df['season'] == 'verão')

for season in (primavera, inverno, outono, verão):
  train_index, validation_index = get_split(df[season & ~test_split], rate=.65)
  splits['train'] = splits['train'].drop(validation_index)
  splits['validation'] = splits['validation'].drop(train_index)
  

In [19]:
escritos = data.save_splits(splits, df)
for caminho, n in escritos.items():
    print(f"  {caminho.relative_to(paths.ROOT)!s:34} {n:6} linhas")


  data\splited\test.csv                 354 linhas
  data\splited\train.csv               1820 linhas
  data\splited\validation.csv           980 linhas
  data\splited\test\ifpee1.csv          354 linhas
  data\splited\test\ifpee2.csv          354 linhas
  data\splited\test\ifpee3.csv          354 linhas
